# 01 · Data profiling — NYC 311 Service Requests

**Goal:** measure the quality of one month of NYC 311 service requests along four classic
data-quality dimensions — **completeness, uniqueness, validity and consistency** — and turn
each rule into a number we can monitor over time.

**Statistics behind it:** every quality rule is a yes/no check on each record, i.e. a
**Bernoulli trial**. The share of records that pass a rule is the estimate p̂ of the
probability that a record is "good". Later notebooks use these proportions to build alerts
(notebook 02), service-level metrics (notebook 03) and audit samples (notebook 04).

**Data:** [311 Service Requests from 2020 to Present](https://data.cityofnewyork.us/d/erm2-nwe9) — NYC Open Data (dataset id `erm2-nwe9`).

## 1. Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

pd.set_option("display.max_columns", 50)
plt.rcParams["figure.figsize"] = (10, 5)

DATA_DIR = Path("../data/raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load one month of data

We download a **small slice** (one month) through the Socrata API instead of the full dataset,
which has tens of millions of rows. The file is cached in `data/raw/`, so the API is only
called once. Change `START` / `END` to profile another month.

In [ ]:
API_URL = "https://data.cityofnewyork.us/resource/erm2-nwe9.json"
START, END = "2025-01-01", "2025-02-01"   # one month
PAGE_SIZE = 50_000
CACHE_FILE = DATA_DIR / f"nyc311_{START[:7]}.csv"


def download_month(start: str, end: str) -> pd.DataFrame:
    """Download all requests created in [start, end) page by page."""
    pages, offset = [], 0
    while True:
        params = {
            "$where": f"created_date >= '{start}T00:00:00' AND created_date < '{end}T00:00:00'",
            "$order": "unique_key",
            "$limit": PAGE_SIZE,
            "$offset": offset,
        }
        resp = requests.get(API_URL, params=params, timeout=120)
        resp.raise_for_status()
        page = resp.json()
        if not page:
            break
        pages.append(pd.DataFrame(page))
        offset += PAGE_SIZE
        print(f"downloaded {offset:,} rows...")
    return pd.concat(pages, ignore_index=True)


if CACHE_FILE.exists():
    df = pd.read_csv(CACHE_FILE, low_memory=False)
else:
    df = download_month(START, END)
    df.to_csv(CACHE_FILE, index=False)

print(f"{len(df):,} rows x {df.shape[1]} columns")

## 3. First look

In [ ]:
# keep the columns used in this notebook (the API may return more)
COLS = ["unique_key", "created_date", "closed_date", "agency", "complaint_type",
        "descriptor", "incident_zip", "incident_address", "city", "borough",
        "status", "latitude", "longitude"]
df = df[[c for c in COLS if c in df.columns]].copy()

for c in ["created_date", "closed_date"]:
    df[c] = pd.to_datetime(df[c], errors="coerce")
for c in ["latitude", "longitude"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df.head()

In [ ]:
df.dtypes

## 4. Completeness — how much is missing?

For each column, every record is a Bernoulli trial: *filled* (1) or *missing* (0).
The completeness rate is the sample proportion p̂ = filled / total.
Empty strings are treated as missing too.

In [ ]:
filled = df.replace(r"^\s*$", np.nan, regex=True).notna()
completeness = (filled.mean() * 100).sort_values()

ax = completeness.plot.barh(color="steelblue")
ax.axvline(95, color="red", linestyle="--", label="95% target")
ax.set_xlabel("% of records filled")
ax.set_title("Completeness by column")
ax.legend()
plt.tight_layout()
plt.show()

completeness.round(2).to_frame("completeness_%")

**Reading the chart:** columns under the 95% line are candidates for a data-quality rule.
Note that some gaps are *expected* — `closed_date` is empty while a request is still open —
so a completeness rule must always be read together with business context.

## 5. Uniqueness — duplicates

Two levels:
1. **Technical duplicates:** the same `unique_key` appears more than once (should be zero).
2. **Business duplicates:** different requests about the *same problem* — same complaint type,
   same address, same day. In 311 data this is common: several neighbours report the same issue.

In [ ]:
tech_dups = df["unique_key"].duplicated().sum()
print(f"Technical duplicates (unique_key): {tech_dups:,}")

df["created_day"] = df["created_date"].dt.date
key = ["complaint_type", "incident_address", "created_day"]
has_address = df["incident_address"].notna()
biz_dup_mask = df[has_address].duplicated(subset=key, keep="first")
biz_dups = biz_dup_mask.sum()
print(f"Possible business duplicates: {biz_dups:,} "
      f"({biz_dups / has_address.sum():.1%} of requests with an address)")

(df[has_address][df[has_address].duplicated(subset=key, keep=False)]
   .groupby("complaint_type").size().sort_values(ascending=False).head(10)
   .to_frame("requests_in_duplicate_groups"))

## 6. Validity — does each value follow the rules?

Each rule below is a yes/no test per record. The pass rate is again a proportion p̂.

In [ ]:
# NYC rough bounding box
LAT_MIN, LAT_MAX, LON_MIN, LON_MAX = 40.49, 40.92, -74.27, -73.68
VALID_BOROUGHS = {"MANHATTAN", "BROOKLYN", "QUEENS", "BRONX", "STATEN ISLAND"}

zip_str = df["incident_zip"].astype("string").str.strip()
has_coords = df["latitude"].notna() & df["longitude"].notna()
is_closed = df["closed_date"].notna()

rules = {
    # (dimension, records checked, records passing)
    "ZIP code has 5 digits": (
        "Validity", zip_str.notna(), zip_str.str.fullmatch(r"\d{5}", na=False)),
    "Coordinates inside NYC": (
        "Validity", has_coords,
        has_coords & df["latitude"].between(LAT_MIN, LAT_MAX)
                   & df["longitude"].between(LON_MIN, LON_MAX)),
    "Borough is a known value": (
        "Validity", df["borough"].notna(),
        df["borough"].str.upper().isin(VALID_BOROUGHS)),
    "Closed date after created date": (
        "Consistency", is_closed, is_closed & (df["closed_date"] >= df["created_date"])),
    "Closed requests have status 'Closed'": (
        "Consistency", is_closed, is_closed & (df["status"].str.lower() == "closed")),
}

scorecard = pd.DataFrame([
    {"rule": name, "dimension": dim,
     "checked": int(checked.sum()),
     "passed": int((passed & checked).sum())}
    for name, (dim, checked, passed) in rules.items()
])
scorecard["pass_rate_%"] = (scorecard["passed"] / scorecard["checked"] * 100).round(2)
scorecard

**Why "checked" differs from the total:** a rule is only applied where it makes sense —
e.g. "closed date after created date" only for closed requests. This is a conditional
proportion: P(rule passes | rule applies).

## 7. Quality scorecard

One table with every rule — the kind of view a data-governance dashboard (e.g. in Power BI)
would track month by month.

In [ ]:
completeness_rules = pd.DataFrame({
    "rule": [f"{c} is filled" for c in completeness.index],
    "dimension": "Completeness",
    "checked": len(df),
    "passed": (completeness.values / 100 * len(df)).round().astype(int),
})
completeness_rules["pass_rate_%"] = completeness.values.round(2)

uniqueness_rule = pd.DataFrame([{
    "rule": "unique_key is unique", "dimension": "Uniqueness",
    "checked": len(df), "passed": len(df) - int(tech_dups),
    "pass_rate_%": round((len(df) - tech_dups) / len(df) * 100, 2),
}])

full_scorecard = (pd.concat([completeness_rules, uniqueness_rule, scorecard], ignore_index=True)
                    .sort_values("pass_rate_%"))
full_scorecard.to_csv("../data/scorecard_01.csv", index=False)
full_scorecard

## 8. Preview for the next notebooks

- **Volume per day** → notebook 02 checks whether a day is "abnormal" (Poisson / normal, z-score).
- **Resolution time** → notebook 03 models time-to-close (exponential) and SLA risk.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

daily = df.groupby("created_day").size()
daily.plot(ax=axes[0], marker="o")
axes[0].set_title("Requests per day")
axes[0].set_ylabel("requests")

hours = (df["closed_date"] - df["created_date"]).dt.total_seconds() / 3600
hours[(hours >= 0) & (hours <= 24 * 14)].plot.hist(bins=60, ax=axes[1], color="gray")
axes[1].set_title("Hours to close (first 14 days)")
axes[1].set_xlabel("hours")

plt.tight_layout()
plt.show()

## 9. Key findings

*Write 3–5 bullet points here after running the notebook, for example:*

- Columns below 95% completeness: …
- Share of possible business duplicates: … — the most duplicated complaint types are …
- Validity rules with the lowest pass rate: …
- Next step: monitor these rates month by month and alert when they drop (notebook 02).